# Exploratory Data Profiling: 2014-2016 Ebola Outbreak

This notebook documents the exploratory investigation and validation process used to understand the WHO / HDX Ebola cases and deaths dataset. It is designed to preserve the data discovery path: what was checked, why it mattered, and how those checks shaped the working dataset.

This notebook is exploratory in nature and should not be interpreted as the project's final analytical report. A later analysis notebook will use the cleaned dataset and documented methodology to tell a more focused story with fewer exploratory detours. Findings here are preliminary and should be treated as evidence for follow-up analysis, not final conclusions.


## 1. Introduction

The project investigates the 2014-2016 Ebola outbreak using the HDX Ebola dataset and supporting public health sources. This notebook focuses on dataset profiling, validation, cleaning decisions, and early exploratory patterns.

The core source is a long-format dataset where each row contains a country, date, indicator, and value. Because the meaning of each value depends on the indicator, the notebook first investigates structure and data quality before building the country-level cases and deaths dataset.


## 2. Project Objectives

Exploratory objectives for this notebook:

1. Confirm the raw dataset structure, date range, and expected unit of analysis.
2. Identify the indicators most relevant to cumulative cases and deaths.
3. Investigate country labels, especially `Liberia 2` and `Guinea 2`.
4. Check missing values, duplicates, and country-date consistency.
5. Build a reproducible `main_countries` dataset with one row per country-date.
6. Validate the reshaped dataset against known totals where appropriate.
7. Explore preliminary patterns in country burden, case fatality rate, timelines, and the relationship between cases and deaths.

These objectives support later project questions about outbreak severity, country-level burden, timelines, interventions, and response lessons. They do not attempt to make causal claims.


In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

sns.set_theme(style="whitegrid", context="notebook")

# Resolve the project root whether the notebook is launched from the project root
# or directly from notebooks.
PROJECT_ROOT = next(
    root for root in [Path.cwd(), *Path.cwd().parents]
    if (root / "datasets" / "raw" / "ebola_data_db_format.csv").exists()
)

RAW_DATA_FILE = PROJECT_ROOT / "datasets" / "raw" / "ebola_data_db_format.csv"
PROCESSED_DATA_FILE = PROJECT_ROOT / "datasets" / "processed" / "master_cases_deaths.csv"
INDICATOR_INVENTORY_FILE = PROJECT_ROOT / "exports" / "reports" / "indicator_inventory.csv"
INDICATOR_INVENTORY_FILE.parent.mkdir(parents=True, exist_ok=True)

RAW_DATA_FILE


## 3. Dataset Overview

The raw HDX file is loaded first without altering the original file in `datasets/raw/`. Dates are parsed for profiling and later time-series exploration. This overview confirms the basic shape, columns, and reporting period before any filtering decisions are made.


In [ ]:
df = pd.read_csv(RAW_DATA_FILE)
df["Date"] = pd.to_datetime(df["Date"])

print(f"Dataset path: {RAW_DATA_FILE}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")

display(df.head())


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()


In [ ]:
dataset_overview = pd.DataFrame({
    "rows": [len(df)],
    "columns": [df.shape[1]],
    "countries": [df["Country"].nunique()],
    "indicators": [df["Indicator"].nunique()],
    "start_date": [df["Date"].min()],
    "end_date": [df["Date"].max()],
})

dataset_overview


## 4. Data Quality Assessment

Before selecting indicators or reshaping the data, this section checks missing values, duplicate rows, and the distribution of the numeric `value` field. The expected raw grain is one row per `Country` / `Date` / `Indicator`.


In [ ]:
missing_summary = (
    df.isna()
    .sum()
    .rename("missing_rows")
    .to_frame()
)

missing_summary["missing_percent"] = (
    missing_summary["missing_rows"] / len(df) * 100
).round(2)

missing_summary


In [ ]:
raw_duplicate_count = df.duplicated(subset=["Country", "Date", "Indicator"]).sum()

print(f"Duplicate Country-Date-Indicator rows: {raw_duplicate_count:,}")

duplicate_rows = df[
    df.duplicated(subset=["Country", "Date", "Indicator"], keep=False)
].sort_values(["Country", "Date", "Indicator"])

display(duplicate_rows.head())


In [ ]:
value_summary = df["value"].describe().to_frame()
value_summary


The initial quality checks help validate that the source can support a country-date-indicator workflow. Any missing values or duplicate records identified here would need to be carried into the cleaning decisions before analysis.


## 5. Indicator Investigation

The HDX file contains many indicators beyond the cumulative cases and deaths needed for the main country-level analysis. Listing and inventorying the indicators helps make the filtering decision visible and reproducible.


In [ ]:
indicators = sorted(df["Indicator"].dropna().unique())

print(f"Number of unique indicators: {len(indicators)}")
for indicator in indicators:
    print(f"- {indicator}")


In [ ]:
indicator_inventory = (
    df["Indicator"]
    .value_counts()
    .rename_axis("Indicator")
    .reset_index(name="row_count")
    .sort_values("Indicator")
)

indicator_inventory.to_csv(INDICATOR_INVENTORY_FILE, index=False)

print(f"Saved indicator inventory to: {INDICATOR_INVENTORY_FILE}")
display(indicator_inventory)


In [ ]:
CASE_INDICATOR = "Cumulative number of confirmed, probable and suspected Ebola cases"
DEATH_INDICATOR = "Cumulative number of confirmed, probable and suspected Ebola deaths"

selected_indicators = [CASE_INDICATOR, DEATH_INDICATOR]

indicator_inventory[
    indicator_inventory["Indicator"].isin(selected_indicators)
]


The selected indicators use cumulative confirmed, probable, and suspected cases and deaths. This definition is useful for broad outbreak-burden exploration, but it should be documented because it is not the same as confirmed-only analysis.


## 6. Country Investigation

Country labels define the geographic grain of the analysis. This section checks all country labels and investigates continuation labels before constructing the main country-level dataset.


In [ ]:
country_counts = (
    df["Country"]
    .value_counts()
    .rename_axis("Country")
    .reset_index(name="row_count")
    .sort_values("Country")
)

print(f"Number of country labels in raw data: {df['Country'].nunique()}")
display(country_counts)


In [ ]:
continuation_labels = ["Guinea 2", "Liberia 2"]
continuation_rows = df[df["Country"].isin(continuation_labels)].copy()

continuation_summary = (
    continuation_rows
    .groupby("Country")
    .agg(
        first_date=("Date", "min"),
        last_date=("Date", "max"),
        rows=("value", "size"),
        indicators=("Indicator", "nunique"),
    )
)

print("Continuation-label summary:")
display(continuation_summary)

print("Example continuation rows:")
display(
    continuation_rows
    .sort_values(["Country", "Date", "Indicator"])
    .head(20)
)


The continuation labels appear to be real source records rather than typos. They require a deliberate methodology decision because treating them as ordinary duplicate country names can change the country-date grain.


## 7. Dataset Construction

This section extracts cumulative cases and deaths, checks whether those indicators share the same country-date keys, and reshapes the long-format records into a country-date table.


In [ ]:
cases = df[df["Indicator"].eq(CASE_INDICATOR)].copy()
deaths = df[df["Indicator"].eq(DEATH_INDICATOR)].copy()

print(f"Cases rows: {len(cases):,}")
print(f"Deaths rows: {len(deaths):,}")

display(cases.head())
display(deaths.head())


In [ ]:
case_duplicate_count = cases.duplicated(subset=["Country", "Date"]).sum()
death_duplicate_count = deaths.duplicated(subset=["Country", "Date"]).sum()

print(f"Duplicate country-date rows in cases: {case_duplicate_count:,}")
print(f"Duplicate country-date rows in deaths: {death_duplicate_count:,}")

case_keys = set(zip(cases["Country"], cases["Date"]))
death_keys = set(zip(deaths["Country"], deaths["Date"]))

only_in_cases = case_keys - death_keys
only_in_deaths = death_keys - case_keys

print(f"Country-date keys only in cases: {len(only_in_cases):,}")
print(f"Country-date keys only in deaths: {len(only_in_deaths):,}")


In [ ]:
if only_in_deaths:
    death_only_rows = deaths[
        deaths.set_index(["Country", "Date"]).index.isin(only_in_deaths)
    ].sort_values(["Country", "Date"])
    display(death_only_rows)
else:
    print("No country-date keys appear only in deaths.")


The death-only records suggest that the case and death indicators do not match perfectly on country-date keys. An outer merge preserves these records for validation instead of silently dropping them.


In [ ]:
master_wide = pd.merge(
    cases,
    deaths,
    on=["Country", "Date"],
    how="outer",
    suffixes=("_cases", "_deaths"),
)

master_wide = master_wide.rename(
    columns={
        "value_cases": "cases",
        "value_deaths": "deaths",
    }
)

master_wide = master_wide[
    ["Indicator_cases", "Country", "Date", "cases", "Indicator_deaths", "deaths"]
].sort_values(["Country", "Date"])

print(f"Merged shape: {master_wide.shape[0]:,} rows x {master_wide.shape[1]:,} columns")
display(master_wide.head())


In [ ]:
merge_missing_summary = (
    master_wide[["cases", "deaths"]]
    .isna()
    .sum()
    .rename("missing_rows")
    .to_frame()
)

merge_missing_summary["missing_percent"] = (
    merge_missing_summary["missing_rows"] / len(master_wide) * 100
).round(2)

merge_missing_summary


## 8. Validation Checks

Validation checks focus on whether the reshaped dataset preserves a clean country-date grain and whether continuation labels can be merged safely. This is where the `Liberia 2` and `Guinea 2` decision is tested rather than assumed.


In [ ]:
continuation_case_death_summary = (
    master_wide[master_wide["Country"].isin(continuation_labels)]
    .groupby("Country")
    .agg(
        first_date=("Date", "min"),
        last_date=("Date", "max"),
        rows=("Date", "size"),
        min_cases=("cases", "min"),
        max_cases=("cases", "max"),
        min_deaths=("deaths", "min"),
        max_deaths=("deaths", "max"),
    )
)

continuation_case_death_summary


In [ ]:
merge_test = master_wide.copy()
merge_test["Country"] = merge_test["Country"].replace({
    "Liberia 2": "Liberia",
    "Guinea 2": "Guinea",
})

merged_label_duplicate_count = merge_test.duplicated(subset=["Country", "Date"]).sum()

print(
    "Duplicate country-date rows after directly merging continuation labels: "
    f"{merged_label_duplicate_count:,}"
)

display(
    merge_test[
        merge_test.duplicated(subset=["Country", "Date"], keep=False)
    ]
    .sort_values(["Country", "Date"])
    .head(20)
)


Directly merging `Liberia 2` into `Liberia` and `Guinea 2` into `Guinea` creates duplicate country-date rows. This suggests that the continuation labels should not be merged directly for the main country-level dataset. For this exploratory project, they are excluded from `main_countries` to preserve one row per country-date.


In [ ]:
main_countries = master_wide[
    ~master_wide["Country"].isin(continuation_labels)
].copy()

main_countries = main_countries.sort_values(["Country", "Date"])
main_countries.to_csv(PROCESSED_DATA_FILE, index=False)

print(f"Saved cleaned country-level dataset to: {PROCESSED_DATA_FILE}")
print(f"Rows: {len(main_countries):,}")
print(f"Countries: {main_countries['Country'].nunique():,}")
print(
    "Duplicate country-date rows: "
    f"{main_countries.duplicated(subset=['Country', 'Date']).sum():,}"
)

display(main_countries.head())


In [ ]:
expected_main_countries = {
    "Guinea",
    "Liberia",
    "Sierra Leone",
    "Nigeria",
    "Mali",
    "Senegal",
    "Italy",
    "Spain",
    "United Kingdom",
    "United States of America",
}

observed_main_countries = set(main_countries["Country"].unique())

print("Observed countries:")
print(sorted(observed_main_countries))

print("\nMissing expected countries:", sorted(expected_main_countries - observed_main_countries))
print("Unexpected countries:", sorted(observed_main_countries - expected_main_countries))


This validation step confirms that the cleaned `main_countries` dataset contains the expected country labels and preserves the intended country-date grain. The exclusion of `Liberia 2` and `Guinea 2` should remain visible in the methodology because it affects exact final totals.


## 9. Country-Level Summary Analysis

The country-level summary uses the maximum cumulative case and death values for each country. This provides a preliminary view of outbreak burden after the continuation-label decision has been applied.


In [ ]:
country_summary = (
    main_countries
    .groupby("Country")
    .agg(
        total_cases=("cases", "max"),
        total_deaths=("deaths", "max"),
        first_report=("Date", "min"),
        last_report=("Date", "max"),
        reporting_rows=("Date", "size"),
    )
)

country_summary["cfr_percent"] = (
    country_summary["total_deaths"]
    / country_summary["total_cases"]
    * 100
)

country_summary = country_summary.sort_values("total_cases", ascending=False)
country_summary


In [ ]:
reference_totals = pd.DataFrame(
    {
        "Country": ["Guinea", "Liberia", "Sierra Leone"],
        "reference_cases": [3814, 10678, 14124],
        "reference_deaths": [2544, 4810, 3956],
    }
)

validation_comparison = (
    country_summary
    .reset_index()
    .merge(reference_totals, on="Country", how="inner")
)

validation_comparison["case_difference"] = (
    validation_comparison["total_cases"]
    - validation_comparison["reference_cases"]
)
validation_comparison["death_difference"] = (
    validation_comparison["total_deaths"]
    - validation_comparison["reference_deaths"]
)

validation_comparison[
    [
        "Country",
        "total_cases",
        "reference_cases",
        "case_difference",
        "total_deaths",
        "reference_deaths",
        "death_difference",
    ]
]


The cleaned dataset is close to commonly cited final public totals for the three hardest-hit countries, but it does not match them exactly. This appears consistent with reporting revisions, retrospective corrections, indicator definitions, and the decision to exclude continuation labels from the main country-level dataset.


In [ ]:
plot_country_summary = country_summary.reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cases_plot = plot_country_summary.sort_values("total_cases", ascending=True)
sns.barplot(
    data=cases_plot,
    x="total_cases",
    y="Country",
    ax=axes[0],
    color="#4C78A8",
)
axes[0].set_title("Total Ebola Cases by Country")
axes[0].set_xlabel("Cumulative cases")
axes[0].set_ylabel("Country")

deaths_plot = plot_country_summary.sort_values("total_deaths", ascending=True)
sns.barplot(
    data=deaths_plot,
    x="total_deaths",
    y="Country",
    ax=axes[1],
    color="#F58518",
)
axes[1].set_title("Total Ebola Deaths by Country")
axes[1].set_xlabel("Cumulative deaths")
axes[1].set_ylabel("Country")

plt.tight_layout()
plt.show()


In [ ]:
minor_outbreaks = (
    plot_country_summary
    .query("total_cases < 100")
    .sort_values("total_cases", ascending=True)
)

plt.figure(figsize=(9, 4))
sns.barplot(
    data=minor_outbreaks,
    x="total_cases",
    y="Country",
    color="#72B7B2",
)
plt.title("Countries with Fewer Than 100 Reported Cases")
plt.xlabel("Cumulative cases")
plt.ylabel("Country")
plt.tight_layout()
plt.show()


These summaries suggest that Sierra Leone, Liberia, and Guinea account for most reported cases and deaths in the cleaned country-level dataset. The small-country chart helps keep low-count outbreaks visible without overstating their role in the overall burden.


## 10. Case Fatality Rate Exploration

Case fatality rate is calculated as cumulative deaths divided by cumulative cases. Because countries with very small outbreaks can produce unstable percentages, the chart labels include total case counts.


In [ ]:
cfr_plot_data = (
    country_summary
    .reset_index()
    .sort_values("cfr_percent", ascending=False)
    .copy()
)

cfr_plot_data["Country_Label"] = (
    cfr_plot_data["Country"]
    + " ("
    + cfr_plot_data["total_cases"].astype(int).astype(str)
    + " cases)"
)

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=cfr_plot_data,
    x="cfr_percent",
    y="Country_Label",
    color="#54A24B",
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3)

plt.title("Case Fatality Rate by Country")
plt.xlabel("Deaths as a percent of cumulative cases")
plt.ylabel("Country")
plt.xlim(0, max(cfr_plot_data["cfr_percent"]) * 1.15)
plt.tight_layout()
plt.show()


Mali has the highest case fatality rate in this dataset, but it had only eight reported cases. This result should be interpreted carefully. Countries such as Liberia, Sierra Leone, and Guinea had much larger outbreaks and therefore drove more of the overall mortality burden.


## 11. Timeline Exploration

The time-series exploration focuses on Sierra Leone, Liberia, and Guinea because they experienced the highest case counts. These views help identify growth, peaks, and plateau periods for later comparison with intervention timelines.


In [ ]:
main_countries = main_countries.sort_values(["Country", "Date"])

main_countries["new_cases"] = (
    main_countries
    .groupby("Country")["cases"]
    .diff()
)

main_countries["new_deaths"] = (
    main_countries
    .groupby("Country")["deaths"]
    .diff()
)

top3_countries = ["Guinea", "Liberia", "Sierra Leone"]
top3 = main_countries[main_countries["Country"].isin(top3_countries)].copy()

print("Three hardest-hit countries included in timeline:")
print(sorted(top3["Country"].unique()))


In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=top3,
    x="Date",
    y="cases",
    hue="Country",
)
plt.title("Cumulative Ebola Cases Over Time: Three Hardest-Hit Countries")
plt.xlabel("Date")
plt.ylabel("Cumulative cases")
plt.legend(title="Country")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=top3,
    x="Date",
    y="new_cases",
    hue="Country",
)
plt.title("Reported New Cases Between Reporting Dates")
plt.xlabel("Date")
plt.ylabel("Change in cumulative cases")
plt.legend(title="Country")
plt.tight_layout()
plt.show()


In [ ]:
peak_new_cases = top3.loc[
    top3.groupby("Country")["new_cases"].idxmax(),
    ["Country", "Date", "new_cases"]
].sort_values("new_cases", ascending=False)

peak_new_cases


In [ ]:
top3 = top3.sort_values(["Country", "Date"])
top3["new_cases_smooth_4_period"] = (
    top3
    .groupby("Country")["new_cases"]
    .transform(lambda values: values.rolling(4, min_periods=1).mean())
)

plt.figure(figsize=(12, 6))
sns.lineplot(
    data=top3,
    x="Date",
    y="new_cases_smooth_4_period",
    hue="Country",
)
plt.title("Smoothed Reported New Cases: Four-Period Rolling Average")
plt.xlabel("Date")
plt.ylabel("Average change in cumulative cases")
plt.legend(title="Country")
plt.tight_layout()
plt.show()


The timeline suggests that Liberia experienced a rapid rise and earlier plateau, while Sierra Leone eventually reached the highest cumulative case count in this cleaned dataset. These patterns require careful interpretation because reporting intervals, cumulative reporting, and retrospective revisions can affect period-to-period changes.


## 12. Correlation Exploration

A scatter plot helps check whether countries with more reported cases also had more reported deaths. Smaller countries remain visible as points, but only Guinea, Liberia, and Sierra Leone are annotated to avoid label overlap.


In [ ]:
scatter_data = country_summary.reset_index().copy()
major_countries = ["Guinea", "Liberia", "Sierra Leone"]

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=scatter_data,
    x="total_cases",
    y="total_deaths",
    s=80,
    color="#4C78A8",
)

for _, row in scatter_data[scatter_data["Country"].isin(major_countries)].iterrows():
    plt.text(
        row["total_cases"] + 150,
        row["total_deaths"] + 60,
        row["Country"],
        fontsize=9,
    )

plt.title("Relationship Between Total Cases and Total Deaths")
plt.xlabel("Cumulative cases")
plt.ylabel("Cumulative deaths")
plt.tight_layout()
plt.show()


In [ ]:
cases_deaths_correlation = country_summary[
    ["total_cases", "total_deaths"]
].corr().loc["total_cases", "total_deaths"]

print(f"Correlation between total cases and total deaths: {cases_deaths_correlation:.3f}")


Total cases and total deaths appear strongly positively related within this dataset, although the relationship is heavily influenced by the three largest outbreaks. This should not be interpreted as evidence that mortality risk was identical across countries.


## 13. Preliminary Findings

These are exploratory findings to carry into later analysis, not final project conclusions:

1. The HDX source is long-format and requires indicator filtering before cases and deaths can be analyzed together.
2. The selected cumulative confirmed, probable, and suspected case/death indicators provide a useful base for country-level burden exploration, while also requiring clear definition notes.
3. `Liberia 2` and `Guinea 2` appear to be continuation reporting labels. Directly merging them into the main country labels creates duplicate country-date rows, so they are excluded from `main_countries` for the main country-level dataset.
4. Sierra Leone, Liberia, and Guinea appear to account for most cumulative cases and deaths in the cleaned dataset.
5. Case fatality rates vary by country, but rates for very small outbreaks require caution because their denominators are small.
6. Total cases and total deaths appear strongly positively related within this dataset, although that relationship is heavily shaped by the three largest outbreaks.


## 14. Limitations

Limitations to carry forward:

1. The HDX data may include reporting revisions, retrospective corrections, reclassification, and changes in data sources over time.
2. The analysis uses a cumulative reporting structure. Period-to-period changes are calculated from cumulative totals and may be affected by reporting gaps or retrospective updates.
3. The selected indicators include confirmed, probable, and suspected cases and deaths. They should not be interpreted as confirmed-only counts.
4. `Liberia 2` and `Guinea 2` are excluded from the main country-level dataset to preserve one row per country-date. This decision affects exact totals and should remain documented.
5. Countries with very small case counts can have misleadingly high or low case fatality rates because small denominators make percentages unstable.
6. Dataset trends alone cannot explain intervention effects or causal mechanisms. Later analysis should connect trends to WHO, CDC, academic, and other credible public health sources.


## 15. Publication Status and Handoff

This exploratory notebook is complete and serves as the methodological foundation for the final analysis notebook. The main cleaning decisions, including exclusion of `Liberia 2` and `Guinea 2` from `main_countries`, have been carried into the project documentation and final notebook.

Completed handoff items:

1. Cleaning documentation was updated to reflect the continuation-label decision.
2. The cleaned country-level dataset was connected to source-backed intervention context.
3. Nigeria was developed as a documented contrast case using dataset summaries and external public health sources.
4. `notebooks/02_ebola_outbreak_analysis.ipynb` was created as the publication-oriented final analysis notebook.

Remaining publication work should focus on rendered-output QA and repository packaging rather than additional exploratory analysis.
